# PySnooper mutant statistics per approach

In [ ]:
import csv
import re
from pathlib import Path

import pandas as pd

PROJECT = "PySnooper"
RESULTS_DIR = Path("results") / PROJECT

In [ ]:
ERRORS_BANNER = re.compile(r"^=+ ERRORS =+$", re.M)

def count(text, word):
    """First 'N <word>' integer in the pytest summary (0 if absent)."""
    m = text and re.search(rf"(\d+) {word}\b", text)
    return int(m.group(1)) if m else 0


def classify(survives, text):
    if survives:
        return "survived"
    if not text or ERRORS_BANNER.search(text):
        return "incompetence"
    return "failing_test" if count(text, "failed") else "incompetence"

In [23]:
def collect_mutants(results_dir):
    """One row per mutant, with its kill classification, over every run found."""
    rows = []
    for csv_path in sorted(results_dir.glob("**/mutant_summary.csv")):
        mutants_dir = csv_path.parent                      # <approach>-mutants
        approach = mutants_dir.name.replace("-mutants", "")
        test_dir = mutants_dir.parent / f"{approach}-test"
        config = mutants_dir.parent.parent.name
        for r in csv.DictReader(csv_path.open(newline="")):
            tf = test_dir / f"{r['MUTANT']}_test.txt"
            text = tf.read_text() if tf.exists() else None
            survives = r["SURVIVES"].strip() == "True"
            rows.append({
                "config": config,
                "approach": approach,
                "mutant": r["MUTANT"],
                "survives": survives,
                "failed": count(text, "failed"),
                "errors": count(text, "errors?"),
                "outcome": classify(survives, text),
            })
    return pd.DataFrame(rows)


mutants = collect_mutants(RESULTS_DIR)
print(f"{len(mutants)} mutants across {mutants['approach'].nunique()} approaches")
mutants

434 mutants across 3 approaches


,config,approach,mutant,survives,failed,errors,outcome
0,config-PySnooper-pycompat-subclasshook-L37,basic,mutant_0.py,False,0,8,incompetence
1,config-PySnooper-pycompat-subclasshook-L37,basic,mutant_4.py,False,0,8,incompetence
2,config-PySnooper-pycompat-subclasshook-L37,basic,mutant_5.py,False,0,8,incompetence
3,config-PySnooper-pycompat-subclasshook-L37,basic,mutant_1.py,False,0,8,incompetence
4,config-PySnooper-pycompat-subclasshook-L37,basic,mutant_6.py,False,0,8,incompetence
...,...,...,...,...,...,...,...
429,config-PySnooper-variables-items-L54,stpa,mutant_2.py,False,0,8,incompetence
430,config-PySnooper-variables-items-L54,stpa,mutant_3.py,False,0,8,incompetence
431,config-PySnooper-variables-items-L54,stpa,mutant_7.py,False,0,8,incompetence
432,config-PySnooper-variables-items-L54,stpa,mutant_8.py,False,0,8,incompetence


In [24]:
def summarise(m):
    t = pd.crosstab(m["approach"], m["outcome"]).reindex(
        columns=["incompetence", "failing_test", "survived"], fill_value=0
    )
    s = pd.DataFrame({
        "mutants": t.sum(axis=1),
        "killed_incompetence": t["incompetence"],
        "killed_failing_test": t["failing_test"],
        "survived": t["survived"],
        "competent": t["failing_test"] + t["survived"],   # == mutants - incompetence
    })
    s.loc["ALL"] = s.sum()
    s["competence_rate"] = (s["competent"] / s["mutants"]).round(3)
    s["mutation_score"] = (s["killed_failing_test"] / s["competent"].replace(0, pd.NA)).round(3)
    return s


summary = summarise(mutants)
summary

,mutants,killed_incompetence,killed_failing_test,survived,competent,competence_rate,mutation_score
approach,,,,,,,
basic,100,54,37,9,46,0.460,0.804
hazop,234,110,89,35,124,0.530,0.718
stpa,100,55,24,21,45,0.450,0.533
ALL,434,219,150,65,215,0.495,0.698
